In [ ]:
# Import required module for counting mutations
from collections import Counter

In [ ]:
# Input: aligned probe target sequences in '.txt' format
# Each line represents one genome sequence
with open('Probe_target.txt', 'r') as f:
        genome_seq = f.read().strip()

# Input: Probe target sequence (must match alignment length)
primer_seq = "acactagccatccttactgcgcttcg".lower()  

In [ ]:
# Split sequences into individual genome entries
genome_lines = genome_seq.splitlines()

# Calculate total length for validation
genome_length = sum(len(line) for line in genome_lines)
probe_length = len(probe_seq)

print(f"Length of genome sequence: {genome_length}")
print(f"Length of probe sequence: {probe_length}")

In [ ]:
# Function to calculate:
# - Mutation rates
# - Mutation types
# - Position-wise mutation distribution
def calculate_mutation_percentage_and_types(genome_segment, probe_seq):
    valid_nucleotides = {'a', 'c', 'g', 't'}
    total_positions = len(genome_segment)
    mutation_count = 0
    mutation_types = Counter()
    mutation_positions = [0] * total_positions
    mutation_positions_with_types = {i: Counter() for i in range(total_positions)}
    valid_positions = 0  
    
    for i in range(total_positions):                                                                    # Skip ambiguous or invalid nucleotides
        if genome_segment[i] not in valid_nucleotides or probe_seq[i] not in valid_nucleotides:
            continue  
        valid_positions += 1

        if probe_seq[i] != genome_segment[i]:                                                          # Identify mismatch (mutation)
            mutation_count += 1
            mutation_type = (probe_seq[i], genome_segment[i])
            mutation_types[mutation_type] += 1
            mutation_positions[i] += 1
            mutation_positions_with_types[i][mutation_type] += 1

    mutation_rate = (mutation_count / valid_positions) * 100 if valid_positions > 0 else 0
    return mutation_rate, mutation_types, mutation_positions, mutation_positions_with_types


# Define high- and moderate-risk mutation counters
high_risk_mutation_types_total = Counter()
moderate_risk_mutation_types_total = Counter()

total_mutation_rate = 0
segment_count = 0
mutation_positions = [0] * len(genome_lines[0])
total_mutation_types = Counter()

all_mutation_positions_with_types = {i: Counter() for i in range(len(genome_lines[0]))}

for line in genome_lines:
    mutation_rate, mutation_types, segment_mutation_positions, segment_mutation_positions_with_types = calculate_mutation_percentage_and_types(line.lower(), probe_seq)

    total_mutation_rate += mutation_rate
    total_mutation_types.update(mutation_types)
    mutation_positions = [x + y for x, y in zip(mutation_positions, segment_mutation_positions)]
    segment_count += 1

    for i in range(len(segment_mutation_positions_with_types)):
        for mutation_type, count in segment_mutation_positions_with_types[i].items():
            all_mutation_positions_with_types[i][mutation_type] += count

            # Probe logic (Moderate-risk= first 5 bases from both 5' and 3' ends; High-risk= Remaining middle region)
            if 5 <= i < len(probe_seq) - 5:
                high_risk_mutation_types_total[mutation_type] += count
            else:
                moderate_risk_mutation_types_total[mutation_type] += count

# Compute mutation rates (normalized) across all sequences
normalized_mutation_rate = total_mutation_rate / segment_count if segment_count > 0 else 0
total_lines = len(genome_lines)

# ================= OUTPUT ================= #

# Mutation Rates
print(f"Mutation Rate: {normalized_mutation_rate:.2f}%")

# Mutation type distribution
print("\nMutation Types:")
for mutation, count in total_mutation_types.items():
    print(f"{mutation[0].upper()} -> {mutation[1].upper()}: {count}")

# High-risk mutations # Position-wise mutation mapping # mutation population frequency
print("\n--- High-Risk Mutation Positions ---")
for i, mutation_counter in all_mutation_positions_with_types.items():
    if i >= len(probe_seq) - 5:
        if sum(mutation_counter.values()) > 0:
            mutation_info = ', '.join([
                f"{mut[0].upper()}->{mut[1].upper()} ({(count/total_lines)*100:.2f}%)"
                for mut, count in mutation_counter.items()
            ])
            print(f"Position {i+1}: {mutation_info}")

# Moderate-risk mutations # Position-wise mutation mapping # mutation population frequency
print("\n--- Moderate-Risk Mutation Positions ---")
for i, mutation_counter in all_mutation_positions_with_types.items():
    if i < len(probe_seq) - 5:
        if sum(mutation_counter.values()) > 0:
            mutation_info = ', '.join([
                f"{mut[0].upper()}->{mut[1].upper()} ({(count/total_lines)*100:.2f}%)"
                for mut, count in mutation_counter.items()
            ])
            print(f"Position {i+1}: {mutation_info}")